# 02 从零实现线性回归

## 本节目标

本节用线性回归跑通第一个完整训练闭环。

你将完成：

- 构造带噪声的线性数据。
- 手动实现 `y_hat = wx + b`。
- 使用 MSELoss 和梯度下降更新参数。
- 使用 `nn.Linear` 复现同一个实验。
- 画出 loss 曲线和拟合结果，并解释实验现象。

## 背景问题

本节关注的问题是：如果数据整体接近一条直线，模型能否通过训练自动找到这条直线？

线性回归适合作为第一个训练闭环，因为它的模型简单、结果直观，但流程完整。后续的 MLP、CNN、LSTM 仍然遵循类似的训练结构。

## 核心概念

- **线性模型**：用 `w` 和 `b` 描述输入与输出之间的线性关系。
- **MSELoss**：均方误差，适合连续值预测。
- **学习率**：控制每次参数更新的步长。
- **训练曲线**：观察 loss 是否整体下降。
- **可复现性**：固定随机种子，让实验结果更稳定。

这里的关键不是公式复杂，而是把训练流程写清楚。

## 最小代码示例：生成并查看数据

这个实验使用合成数据：`y = 3x + 2 + noise`。噪声让数据更接近真实场景，也能提醒我们：训练结果不需要完全等于生成参数。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

x = torch.linspace(-2, 2, 140).view(-1, 1)
noise = 0.35 * torch.randn_like(x)
y = 3.0 * x + 2.0 + noise

print("x shape:", x.shape)
print("y shape:", y.shape)

plt.figure(figsize=(6, 4))
plt.scatter(x.numpy(), y.numpy(), s=18, alpha=0.7)
plt.title("Synthetic linear data")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()

## 最小代码示例：手写预测和 MSE

在使用优化器之前，先用固定参数计算一次预测和 loss，确认 shape 对齐。

In [ ]:
w = torch.tensor([0.0])
b = torch.tensor([0.0])

y_pred = x * w + b
mse = torch.mean((y_pred - y) ** 2)

print("prediction shape:", y_pred.shape)
print("initial mse:", mse.item())

## 完整实验：手动参数训练

下面直接把 `w` 和 `b` 定义为可学习 Tensor。训练循环的结构和真实模型训练一致。

In [ ]:
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.08
losses = []

for epoch in range(120):
    y_pred = x * w + b
    loss = torch.mean((y_pred - y) ** 2)

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()
    losses.append(loss.item())

print(f"learned w={w.item():.3f}, b={b.item():.3f}, final loss={losses[-1]:.4f}")

## 实验观察：loss 曲线与拟合线

从运行结果可以看到，loss 通常会快速下降，然后进入较平稳的区域。因为数据有噪声，最终 loss 不会变成 0。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("Manual linear regression loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

with torch.no_grad():
    fitted = x * w + b

plt.figure(figsize=(6, 4))
plt.scatter(x.numpy(), y.numpy(), s=18, alpha=0.6, label="data")
plt.plot(x.numpy(), fitted.numpy(), color="red", linewidth=2, label="fitted line")
plt.title("Fitted line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 完整实验：使用 `nn.Linear`

实际项目里通常不会手动管理每个参数，而是用 `nn.Module` 和优化器。下面用 `nn.Linear(1, 1)` 完成同样的任务。

In [ ]:
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)
module_losses = []

for epoch in range(120):
    prediction = model(x)
    loss = criterion(prediction, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    module_losses.append(loss.item())

print("weight:", model.weight.item())
print("bias:", model.bias.item())
print("final loss:", module_losses[-1])

## 对比观察

手写参数和 `nn.Linear` 的结果应该接近。两种写法的核心逻辑相同：

- 手写版本帮助理解梯度更新。
- `nn.Linear` 版本更接近真实项目写法。

初学者可以先写手动版本，再换成 `nn.Module`。这样更容易理解框架到底帮我们做了什么。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses, label="manual parameters")
plt.plot(module_losses, label="nn.Linear", linestyle="--")
plt.title("Loss comparison")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 常见错误

1. **学习率太大**：loss 上下震荡，甚至变成 `nan`。  
2. **学习率太小**：loss 下降非常慢。  
3. **忘记清空梯度**：梯度累积导致更新异常。  
4. **预测值和目标值 shape 不一致**：可能触发广播，代码能跑但含义错了。  
5. **只看最终参数，不看曲线**：无法判断训练过程是否稳定。

调试时可以优先检查：`x.shape`、`y.shape`、`prediction.shape`、loss 曲线。

## 概念问答

**Q：为什么线性回归用 MSELoss？**  
A：目标是连续值，MSE 能衡量预测值和真实值之间的平方误差。

**Q：loss 下降是否代表模型一定好？**  
A：只能说明模型在当前训练数据上优化了，还需要看测试数据和任务目标。

**Q：为什么学到的参数不完全等于 3 和 2？**  
A：数据有噪声，训练得到的是当前样本上的较优近似。

**Q：为什么要学习手写版本？**  
A：它能帮助理解参数、梯度和优化器之间的关系。

## 小结

线性回归是最适合入门的训练闭环实验。理解这一章后，可以把模型换成 MLP、CNN 或 LSTM，但训练循环的主干仍然是：前向、loss、反向传播、优化、观察结果。